# Stage 3 — Mars Network Explorer

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 3 — Mars interactive network exploration |
| **Previous stage** | Stage 2 — Earth network exploration (`notebooks/analysis/06_earth_network_explorer.ipynb`) |
| **Next stage** | Stage 4 — Earth-Mars regime calibration (`notebooks/regime/00_calibration_overview.ipynb`) |
| **Purpose** | Browse Martian valley networks on a MOLA hillshade background. Inspect network structure, Strahler order distribution, and drainage density metrics. Provides the visual context for Earth-Mars calibration. |
| **Inputs** | `data/final_valleys/final_valleys.shp` (RAW_KEEP); `data/Mars/MOLA_Hillshade_Robinson_128ppd.tif`; `data/Mars/topology/mars_vn_topology_model_ready.gpkg` |
| **Outputs** | Network maps, summary tables. Nothing written to disk. |
| **Decision gate** | Informational. Use to identify source-data issues before Stage 4 calibration. |

## 0. Configuration

In [ ]:
# Number of example networks to show in the detail panel.
N_NETWORKS_DETAIL = 6

# Show MOLA hillshade as background (requires rasterio).
SHOW_HILLSHADE = True

# Set a specific network ID to highlight (None = show top N by node count).
HIGHLIGHT_NETWORK_ID = None

## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from channel_heads.io.paths import MARS_DIR, MARS_HILLSHADE, FINAL_VALLEYS_DIR
from channel_heads.dd_calibration import mars_network_table, mars_network_strahler

TOPOLOGY_GPKG = MARS_DIR / "topology" / "mars_vn_topology_model_ready.gpkg"
VALLEY_SHP    = FINAL_VALLEYS_DIR / "final_valleys.shp"

print(f"MOLA hillshade : {MARS_HILLSHADE}  exists={MARS_HILLSHADE.exists()}")
print(f"Topology GeoPackage : {TOPOLOGY_GPKG}  exists={TOPOLOGY_GPKG.exists()}")
print(f"Valley shapefile    : {VALLEY_SHP}  exists={VALLEY_SHP.exists()}")

## 2. Load Mars network summary table

In [ ]:
net_table = mars_network_table(TOPOLOGY_GPKG if TOPOLOGY_GPKG.exists() else None)

print(f"Networks loaded: {len(net_table)}")
print()
display(net_table.describe())

In [ ]:
# Sort by drainage density and show top 20
if 'drainage_density_km_km2' in net_table.columns:
    top = net_table.sort_values('drainage_density_km_km2', ascending=False).head(20)
    display(top)
elif 'n_nodes' in net_table.columns:
    top = net_table.sort_values('n_nodes', ascending=False).head(20)
    display(top)
else:
    display(net_table.head(20))

## 3. Drainage density distribution

In [ ]:
dd_col = next((c for c in net_table.columns if 'density' in c.lower()), None)

if dd_col:
    fig, ax = plt.subplots(figsize=(8, 4))
    dd_vals = net_table[dd_col].dropna()
    ax.hist(dd_vals, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(dd_vals.median(), color='red', linestyle='--', label=f'Median {dd_vals.median():.2f}')
    ax.set_xlabel(dd_col)
    ax.set_ylabel('Network count')
    ax.set_title('Mars valley-network drainage density distribution')
    ax.legend()
    fig.tight_layout()
    plt.show()
    print(f"Median DD : {dd_vals.median():.3f} km/km²")
    print(f"Mean DD   : {dd_vals.mean():.3f} km/km²")
else:
    print("No drainage density column found. Available columns:", net_table.columns.tolist())

## 4. Strahler order distribution across all networks

In [ ]:
strahler_df = mars_network_strahler(TOPOLOGY_GPKG if TOPOLOGY_GPKG.exists() else None)

if strahler_df is not None and not strahler_df.empty:
    print("Strahler summary (rows = networks, cols = orders):")
    display(strahler_df.head(10))

    # Aggregate across all networks
    order_cols = [c for c in strahler_df.columns if str(c).isdigit()]
    if order_cols:
        totals = strahler_df[order_cols].sum()
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar([int(c) for c in order_cols], totals.values, color='steelblue', edgecolor='white')
        ax.set_xlabel('Strahler order')
        ax.set_ylabel('Total node count (all networks)')
        ax.set_title('Mars valley-network Strahler order distribution')
        ax.set_xticks([int(c) for c in order_cols])
        fig.tight_layout()
        plt.show()
else:
    print("Strahler data not available or topology GeoPackage not found.")

## 5. Network overview map on MOLA hillshade

In [ ]:
try:
    import geopandas as gpd
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.windows import from_bounds
    HAS_GEO = True
except ImportError as e:
    HAS_GEO = False
    print(f"Geographic packages not available ({e}). Skipping map.")

if HAS_GEO and TOPOLOGY_GPKG.exists() and MARS_HILLSHADE.exists() and SHOW_HILLSHADE:
    import warnings, math
    from pyproj import CRS, Transformer
    from matplotlib.ticker import FuncFormatter
    from channel_heads.dd_calibration import mars_network_table

    _MARS_R = 3_396_190.0

    _nt = mars_network_table(TOPOLOGY_GPKG)
    _dd_col = 'dd_hull_km_km2' if 'dd_hull_km_km2' in _nt.columns else 'length_km'
    _ok = _nt[_nt['status'] == 'ok'].copy()
    _median = _ok[_dd_col].median()
    _half_window = _ok[_dd_col].std() * 0.5
    _near_median = _ok[
        (_ok[_dd_col] >= _median - _half_window) &
        (_ok[_dd_col] <= _median + _half_window)
    ]
    _size_col = 'hull_area_km2' if 'hull_area_km2' in _near_median.columns else 'length_km'
    _top_ids = (
        _near_median
        .sort_values(_size_col, ascending=False)
        .head(N_NETWORKS_DETAIL)['network_id']
        .tolist()
    )
    print(f"Median {_dd_col}: {_median:.3f}  ±{_half_window:.3f}")
    print(f"Candidates in window: {len(_near_median)},  selected: {len(_top_ids)}  (sorted by {_size_col})")

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        topo_gdf = gpd.read_file(TOPOLOGY_GPKG)
    sample_gdf = topo_gdf[topo_gdf['network_id'].isin(_top_ids)].copy()

    with rasterio.open(MARS_HILLSHADE) as _src:
        _rob_crs  = CRS.from_wkt(_src.crs.to_wkt())
        _geog_crs = CRS.from_proj4(f'+proj=longlat +R={_MARS_R} +no_defs')
        _rob2ll   = Transformer.from_crs(_rob_crs, _geog_crs, always_xy=True)
        _ll2rob   = Transformer.from_crs(_geog_crs, _rob_crs, always_xy=True)

        sample_reproj = sample_gdf.to_crs(_rob_crs)

        ncols = min(3, N_NETWORKS_DETAIL)
        nrows = math.ceil(N_NETWORKS_DETAIL / ncols)
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6 * ncols, 5 * nrows),
                                 squeeze=False)

        for ax_i, (_, row) in enumerate(sample_reproj.iterrows()):
            ax = axes[ax_i // ncols][ax_i % ncols]
            geom = row.geometry
            xmin, ymin, xmax, ymax = geom.bounds
            pad_x = max((xmax - xmin) * 0.25, 5_000)
            pad_y = max((ymax - ymin) * 0.25, 5_000)
            cl = max(xmin - pad_x, _src.bounds.left)
            cr = min(xmax + pad_x, _src.bounds.right)
            cb = max(ymin - pad_y, _src.bounds.bottom)
            ct = min(ymax + pad_y, _src.bounds.top)
            cy_rob = (cb + ct) / 2
            cx_rob = (cl + cr) / 2

            win = from_bounds(cl, cb, cr, ct, _src.transform)
            ow = max(1, min(2048, int(round(win.width))))
            oh = max(1, min(2048, int(round(win.height))))
            hs = _src.read(1, window=win, out_shape=(oh, ow),
                           resampling=Resampling.bilinear).astype(float)

            # --- plot MOLA then overlay network ---
            ax.imshow(hs, cmap='gray', extent=[cl, cr, cb, ct],
                      aspect='auto',
                      vmin=np.nanpercentile(hs, 2),
                      vmax=np.nanpercentile(hs, 98))
            gpd.GeoDataFrame(geometry=[geom]).plot(
                ax=ax, color='cyan', linewidth=1.2)

            # --- lock crop window (after geopandas which resets limits) ---
            ax.set_xlim(cl, cr)
            ax.set_ylim(cb, ct)

            # --- lat/lon formatters: convert Robinson metres -> degrees ---
            def _lon_fmt(x, pos, _cy=cy_rob, _t=_rob2ll):
                lon, _ = _t.transform(x, _cy)
                return f'{abs(lon):.2f}°{"E" if lon >= 0 else "W"}'

            def _lat_fmt(y, pos, _cx=cx_rob, _t=_rob2ll):
                _, lat = _t.transform(_cx, y)
                return f'{abs(lat):.2f}°{"N" if lat >= 0 else "S"}'

            ax.xaxis.set_major_formatter(FuncFormatter(_lon_fmt))
            ax.yaxis.set_major_formatter(FuncFormatter(_lat_fmt))
            ax.tick_params(labelsize=6)

            # --- scale bar ---
            _, lat_b = _rob2ll.transform(cx_rob, cb)
            _, lat_t_val = _rob2ll.transform(cx_rob, ct)
            lon_l, _ = _rob2ll.transform(cl, cy_rob)
            dlon = 0.1
            x1r, _ = _ll2rob.transform(lon_l + dlon, (lat_b + lat_t_val) / 2)
            x2r, _ = _ll2rob.transform(lon_l,        (lat_b + lat_t_val) / 2)
            rob_m_per_km = abs(x1r - x2r) / (
                math.radians(dlon) * _MARS_R / 1000.0 *
                math.cos(math.radians((lat_b + lat_t_val) / 2))
            )
            panel_km = (cr - cl) / rob_m_per_km
            sb_km = next(
                k for k in [10, 20, 50, 100, 200, 500, 1000]
                if k < panel_km * 0.35
            )
            sb_len = sb_km * rob_m_per_km
            sb_x0  = cl + (cr - cl) * 0.05
            sb_y0  = cb + (ct - cb) * 0.06
            sb_h   = (ct - cb) * 0.012
            ax.add_patch(plt.Rectangle(
                (sb_x0, sb_y0), sb_len, sb_h,
                facecolor='black', edgecolor='none', zorder=5))
            ax.text(sb_x0 + sb_len / 2, sb_y0 + sb_h * 2.2,
                    f'{sb_km} km', ha='center', va='bottom',
                    fontsize=6, color='black',
                    bbox=dict(facecolor='white', alpha=0.6,
                              edgecolor='none', pad=1),
                    zorder=6)

            nid = row['network_id']
            dd  = _nt.loc[_nt['network_id'] == nid, _dd_col].values[0]
            ax.set_title(f'Network {nid}  ({_dd_col}={dd:.3f})', fontsize=9)

        for ax_i in range(len(sample_reproj), nrows * ncols):
            axes[ax_i // ncols][ax_i % ncols].set_visible(False)

    fig.suptitle(
        f'Mars VNs near median {_dd_col} ({_median:.3f})\n'
        f'MOLA hillshade · Robinson proj. · cyan = valley network',
        fontsize=11)
    fig.tight_layout()
    plt.show()

elif HAS_GEO and TOPOLOGY_GPKG.exists():
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        topo_gdf = gpd.read_file(TOPOLOGY_GPKG)
    fig, ax = plt.subplots(figsize=(14, 7))
    topo_gdf.plot(ax=ax, color='steelblue', linewidth=0.5)
    ax.set_title('Mars valley networks (no hillshade — MOLA not found or SHOW_HILLSHADE=False)')
    fig.tight_layout()
    plt.show()


## 6. Per-network detail — top N networks

Show individual network maps for the largest (by node count or DD) networks.

In [ ]:
if HAS_GEO and TOPOLOGY_GPKG.exists():
    import warnings
    nodes_gdf = None
    edges_gdf = None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            edges_gdf = gpd.read_file(TOPOLOGY_GPKG, layer='edges')
        except Exception:
            pass
        try:
            nodes_gdf = gpd.read_file(TOPOLOGY_GPKG, layer='nodes')
        except Exception:
            pass

    if edges_gdf is not None and 'network_id' in edges_gdf.columns:
        net_id_col = 'network_id'
        top_nets = (
            edges_gdf.groupby(net_id_col).size()
            .sort_values(ascending=False)
            .head(N_NETWORKS_DETAIL)
            .index.tolist()
        )
        if HIGHLIGHT_NETWORK_ID is not None:
            top_nets = [HIGHLIGHT_NETWORK_ID]

        ncols = min(3, len(top_nets))
        nrows = -(-len(top_nets) // ncols)
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(5 * ncols, 4.5 * nrows))
        axes = np.array(axes).flatten()

        for i, nid in enumerate(top_nets):
            ax = axes[i]
            sub = edges_gdf[edges_gdf[net_id_col] == nid]
            sub.plot(ax=ax, color='steelblue', linewidth=1.2)
            if nodes_gdf is not None and net_id_col in nodes_gdf.columns:
                nsub = nodes_gdf[nodes_gdf[net_id_col] == nid]
                outlet = nsub[nsub.get('node_type', pd.Series()) == 'outlet'] if 'node_type' in nsub.columns else None
                if outlet is not None and not outlet.empty:
                    outlet.plot(ax=ax, color='red', markersize=6, zorder=5)
            ax.set_title(f'Network {nid}\n({len(sub)} edges)', fontsize=9)
            ax.axis('off')

        for j in range(len(top_nets), len(axes)):
            axes[j].axis('off')

        fig.suptitle(f'Top {len(top_nets)} Mars networks by edge count', fontsize=11)
        fig.tight_layout()
        plt.show()
    else:
        print("Edges layer not found or no 'network_id' column. Check topology GeoPackage.")
else:
    print("geopandas not available or topology GeoPackage not found. Skipping detail view.")